# KuaiRec-Schema Dataset — Exploratory Data Analysis

Run `python src/data/download_data.py` from the project root first.

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve()
while not (ROOT / 'setup.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

DATA_DIR = ROOT / 'data' / 'raw'
df = pd.read_csv(DATA_DIR / 'interactions.csv')
videos = pd.read_csv(DATA_DIR / 'videos.csv')
print(f'Interactions : {len(df):,}')
print(f'Unique users : {df.user_id.nunique():,}')
print(f'Unique videos: {df.video_id.nunique():,}')
df.head()

## 1. Watch Ratio Distribution and Label Balance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['watch_ratio'], bins=50, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].axvline(0.7, color='tomato', linestyle='--', linewidth=1.5, label='pos_threshold=0.7')
axes[0].set_xlabel('watch_ratio')
axes[0].set_ylabel('Count')
axes[0].set_title('Watch Ratio Distribution')
axes[0].legend()

pos = (df['watch_ratio'] >= 0.7).sum()
neg = len(df) - pos
axes[1].bar(['Negative < 0.7', 'Positive >= 0.7'], [neg, pos],
            color=['#e07b71', '#6aab9c'], edgecolor='white')
axes[1].set_title('Label Balance  pos_rate={:.1f}%'.format(pos/len(df)*100))
axes[1].set_ylabel('Count')
for i, v in enumerate([neg, pos]):
    axes[1].text(i, v + 50, '{:,}'.format(v), ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## 2. User Activity and Item Popularity (Long-tail)

In [ ]:
user_counts = df.groupby('user_id').size().sort_values(ascending=False).values
item_counts = df.groupby('video_id').size().sort_values(ascending=False).values

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].fill_between(range(len(user_counts)), user_counts, alpha=0.4, color='steelblue')
axes[0].plot(user_counts, color='steelblue', linewidth=0.8)
axes[0].set_title('User Activity (Long-tail)')
axes[0].set_xlabel('User rank')
axes[0].set_ylabel('Interactions')

axes[1].fill_between(range(len(item_counts)), item_counts, alpha=0.4, color='darkorange')
axes[1].plot(item_counts, color='darkorange', linewidth=0.8)
axes[1].set_title('Item Popularity (Long-tail)')
axes[1].set_xlabel('Video rank')
axes[1].set_ylabel('Interactions')

plt.tight_layout()
plt.show()

top10 = item_counts[:int(len(item_counts)*0.1)].sum() / item_counts.sum()
print('Top 10% videos account for {:.1f}% of all interactions'.format(top10*100))

## 3. Category and Duration Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

cat_counts = df['video_category'].value_counts().sort_index()
axes[0].bar(cat_counts.index, cat_counts.values, color='mediumpurple', edgecolor='white')
axes[0].set_xlabel('Category ID')
axes[0].set_ylabel('Interactions')
axes[0].set_title('Interactions per Category')

bins = [0, 15, 30, 60, 120, 300]
labels = ['<15s', '15-30s', '30-60s', '60-120s', '>120s']
dur_counts = pd.cut(videos['video_duration'], bins=bins, labels=labels).value_counts().reindex(labels)
axes[1].bar(labels, dur_counts.values, color='#6aab9c', edgecolor='white')
axes[1].set_xlabel('Duration Bucket')
axes[1].set_ylabel('Number of Videos')
axes[1].set_title('Video Duration Distribution')

plt.tight_layout()
plt.show()

## 4. Sparsity and Engagement Actions

In [ ]:
n_users = df['user_id'].nunique()
n_vids  = df['video_id'].nunique()
sparsity = 1 - len(df) / (n_users * n_vids)
print('Matrix : {} x {}'.format(n_users, n_vids))
print('Sparsity: {:.4f}  ({:.2f}% empty)'.format(sparsity, sparsity*100))
print()
for col in ['like', 'follow', 'comment', 'share']:
    print('  {:8s} rate: {:.2f}%'.format(col, df[col].mean()*100))

df['wr_bucket'] = pd.cut(df['watch_ratio'],
                         bins=[0, 0.3, 0.5, 0.7, 1.0, 2.0],
                         labels=['0-30%','30-50%','50-70%','70-100%','>100%'])
eng = df.groupby('wr_bucket')[['like','follow','comment','share']].mean() * 100

fig, ax = plt.subplots(figsize=(10, 4))
eng.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='white')
ax.set_xlabel('Watch Ratio Bucket')
ax.set_ylabel('Engagement Rate (%)')
ax.set_title('Engagement Actions by Watch Depth')
ax.legend(loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()